In [ ]:

import time
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. データの準備（CNN特徴量を模した、ノイズの多い高次元データ）
# ---------------------------------------------------------
# 例えば、本質的には低次元だが、高次元空間に埋め込まれ、大量のノイズが乗ったデータを作成します
# synth_data10000 が既にそのようなデータであればそのまま使用してください

# ---------------------------------------------------------
# 実験A: 失敗例（PCAなしでいきなりt-SNE）
# ---------------------------------------------------------
print("Experiment A: Direct t-SNE on High-Dim Data...")
start = time.time()
# perplexityは通常30-50ですが、高次元だと挙動が不安定になることを示唆
tsne_direct = TSNE(n_components=2, random_state=0).fit_transform(synth_data10000) 
elapsed_a = time.time() - start

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].scatter(tsne_direct[:, 0], tsne_direct[:, 1], c=true_labels, cmap='tab10', s=10)
ax[0].set_title(f"Fail A: Direct t-SNE\n(Time: {elapsed_a:.1f}s)\nNoisy & Slow")

# ---------------------------------------------------------
# 実験B: 失敗例（PCAだけで2次元化）
# ---------------------------------------------------------
print("Experiment B: Direct PCA to 2D...")
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(synth_data10000)

ax[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=true_labels, cmap='tab10', s=10)
ax[1].set_title("Fail B: PCA only to 2D\nOverlapped Clusters")

# ---------------------------------------------------------
# 実験C: 成功例（PCA(50) -> t-SNE）
# ---------------------------------------------------------
print("Experiment C: PCA(50) -> t-SNE...")
start = time.time()
# 1. PCAで「ノイズ」と言える細かい変動を捨て、主要な50次元を取り出す
pca_50 = PCA(n_components=50)
X_pca_50 = pca_50.fit_transform(synth_data10000)

# 2. その情報を使ってt-SNE
tsne_pca = TSNE(n_components=2, random_state=0).fit_transform(X_pca_50)
elapsed_c = time.time() - start

ax[2].scatter(tsne_pca[:, 0], tsne_pca[:, 1], c=true_labels, cmap='tab10', s=10)
ax[2].set_title(f"Success: PCA(50) -> t-SNE\n(Time: {elapsed_c:.1f}s)\nClear Separation")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

# ---------------------------------------------------------
# 1. データ生成： 「干し草の山（大量のノイズ次元）の中の針（少数の信号次元）」
# ---------------------------------------------------------
n_samples = 500
n_total_dims = 2000   # 全次元数（CNNの出力想定）
n_signal_dims = 20    # 実際に意味がある次元数
n_classes = 5         # クラス数

# (A) まず、低次元空間で明確に分かれるクラスごとの塊（Blob）を作る
# これが「本来の信号」です
X_signal, y = make_blobs(n_samples=n_samples, n_features=n_signal_dims, 
                         centers=n_classes, cluster_std=1.0, random_state=42)

# (B) 残りの1980次元はただのノイズ（ガウス分布）
# ノイズの強さをあえて大きくします（scale=5.0）
# これにより、単純なユークリッド距離ではクラスの区別がつかなくなります
n_noise_dims = n_total_dims - n_signal_dims
X_noise = np.random.normal(loc=0, scale=5.0, size=(n_samples, n_noise_dims))

# (C) 信号とノイズを結合して2000次元データにする
X_high_dim = np.hstack([X_signal, X_noise])

# シャッフルして「どの次元が重要か」をわからなくする（PCAに探させるため）
# ※この処理はなくてもPCAは動きますが、よりリアルにするため
perm = np.random.permutation(n_total_dims)
X_high_dim = X_high_dim[:, perm]

print(f"Data Shape: {X_high_dim.shape}")
print(f"Signal Dimensions: {n_signal_dims}, Noise Dimensions: {n_noise_dims}")
print("Note: Noise magnitude is high, drowning out the signal in Euclidean distance.")

# ---------------------------------------------------------
# 2. 比較実験
# ---------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(21, 6))

# --- 実験A: 失敗（PCAのみで2次元化） ---
# 信号が強ければこれでも分かれますが、非線形性が必要な複雑な配置だと重なることがあります。
# 今回は「Blob」なのでPCAだけでもそこそこ分かれるかもしれませんが、
# t-SNEの方が「隙間」をきれいに作れることを後で示します。
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_high_dim)
ax[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
ax[0].set_title("Result A: PCA only (2D)\n(Linear separation)", fontsize=14)

# --- 実験B: 失敗（生のままt-SNE） ---
# ここが重要です。ノイズ次元が多すぎて、点同士の距離がデタラメになり、
# クラスが混ざり合ってしまいます。
print("Running t-SNE on raw data (this may take a while)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='random', learning_rate=200)
X_tsne_raw = tsne.fit_transform(X_high_dim)

ax[1].scatter(X_tsne_raw[:, 0], X_tsne_raw[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
ax[1].set_title("Result B: Raw t-SNE\n(KILLED by Noise Dimensions)", fontsize=14, fontweight='bold', color='red')
# 軸を消す
ax[1].set_xticks([])
ax[1].set_yticks([])

# --- 実験C: 成功（PCAで信号抽出 -> t-SNE） ---
# PCAで「分散が大きい＝情報がある」上位50次元を取り出します。
# これにより1950次元分のノイズがカットされます。
print("Running PCA -> t-SNE...")
pca_50 = PCA(n_components=100)
X_pca_50 = pca_50.fit_transform(X_high_dim)

X_tsne_pca = tsne.fit_transform(X_pca_50)

ax[2].scatter(X_tsne_pca[:, 0], X_tsne_pca[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
ax[2].set_title("Result C: PCA(50) -> t-SNE\n(Noise Removed -> Clear Clusters)", fontsize=14, fontweight='bold', color='green')
ax[2].set_xticks([])
ax[2].set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import manifold, datasets
from sklearn.decomposition import PCA

# ---------------------------------------------------------
# 1. データの準備： 「高次元に埋め込まれた、ノイズの多いS字カーブ」
# ---------------------------------------------------------
n_samples = 1000
n_features = 2000 # 2000次元のデータとする（CNNの特徴量を模倣）
noise_level = 0.1 # ノイズの量

# (A) 本質は3次元の「S字カーブ」データを生成（非線形な構造の代表例）
# color はデータの位置を表す（クラスラベルの代わりとしてグラデーションで表示）
X_struct, color = datasets.make_s_curve(n_samples, random_state=0)

# (B) これを2000次元空間に埋め込む
# まず、ランダムな直交行列を使って3次元の情報を2000次元全体に散らす（回転）
# ※計算を軽くするため、単純にスパースなランダム射影を行います
projection = np.random.randn(n_features, 3) 
X_high_dim = X_struct @ projection.T

# (C) 邪魔なノイズを加える（これが「生のt-SNE」を失敗させる要因）
# データ構造を隠すためのノイズ
X_high_dim += np.random.randn(n_samples, n_features) * noise_level

print(f"Data shape: {X_high_dim.shape} (Samples, Features)")

# ---------------------------------------------------------
# 2. 比較実験
# ---------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

# --- 実験A: 失敗（PCAのみで2次元化） ---
# 線形変換では「S字」の重なりを解消できず、色が混ざって見えるはず
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_high_dim)
ax[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=color, cmap=plt.cm.Spectral, s=10)
ax[0].set_title("Fail A: PCA only (2D)\n(Linear projection smashes the manifold)", fontsize=14)
ax[0].set_xticks([])
ax[0].set_yticks([])

# --- 実験B: 失敗（生のままt-SNE） ---
# 2000次元すべての距離を計算するため、ノイズの影響をモロに受け、構造がぼやける
# （計算時間もかかります）
tsne = manifold.TSNE(n_components=2, init='pca', random_state=0, perplexity=30)
X_tsne_raw = tsne.fit_transform(X_high_dim)
ax[1].scatter(X_tsne_raw[:, 0], X_tsne_raw[:, 1], c=color, cmap=plt.cm.Spectral, s=10)
ax[1].set_title("Fail B: Raw t-SNE\n(Noise in high-dim distracts visualization)", fontsize=14)
ax[1].set_xticks([])
ax[1].set_yticks([])

# --- 実験C: 成功（PCAでノイズ除去 -> t-SNE） ---
# PCAで主要な情報（例えば30次元くらい）を取り出してノイズを捨て、
# その後t-SNEで非線形な「S字」を開く
pca_50 = PCA(n_components=30)
X_pca_50 = pca_50.fit_transform(X_high_dim)

X_tsne_pca = tsne.fit_transform(X_pca_50)
ax[2].scatter(X_tsne_pca[:, 0], X_tsne_pca[:, 1], c=color, cmap=plt.cm.Spectral, s=10)
ax[2].set_title("Success: PCA(30) -> t-SNE\n(Denoised & Unfolded)", fontsize=14, fontweight='bold')
ax[2].set_xticks([])
ax[2].set_yticks([])

plt.tight_layout()
plt.show()